# Merge GDSC CellHit predictions

Build the intermediate cache from published predictions and GDSC web exports.

In [ ]:
import os
from pathlib import Path

project_root = os.environ.get("MSS_TMB_PROJECT_ROOT")
if project_root:
    PROJECT_ROOT = Path(project_root).expanduser().resolve()
else:
    cwd = Path.cwd()
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data").is_dir() and (p / "scripts").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Set MSS_TMB_PROJECT_ROOT to the project directory.")


import hashlib
import json
import os
import re
import sqlite3
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path(os.environ.get('CELLHIT_BASE_DIR', str(PROJECT_ROOT / 'data'))).expanduser()
CELLHIT_DIR = BASE_DIR / 'CellHit'
WEB_GDSC_DIR = CELLHIT_DIR / 'results'

WEB_GDSC_FILES = sorted(WEB_GDSC_DIR.glob('cellhit_gdsc*.csv'))
ORIGINAL_INPUTS = {
    'GDSC': CELLHIT_DIR / 'full_results_gdsc.csv',
    'PRISM': CELLHIT_DIR / 'full_results_prism.csv',
}
OLD_CACHE_DB = CELLHIT_DIR / 'CRC_UCEC_MSI_TMB_RANKINGS' / 'cellhit_matched_patients.sqlite'
CRC_CLINICAL = BASE_DIR / 'clinical_data.csv'
UCEC_CLINICAL = BASE_DIR / 'ucec_clinical.csv'
OUTPUT_DIR = CELLHIT_DIR / 'cellhit_gdsc_rankings'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DB = OUTPUT_DIR / 'cellhit_merged_predictions.sqlite'
CACHE_MANIFEST = OUTPUT_DIR / 'merged_cache_manifest.json'

CHUNK_SIZE = 100_000
REBUILD_CACHE = False
DRUG_ALIASES = {'5-fluorouracil':'fluorouracil','5 fluorouracil':'fluorouracil',
                '5-fu':'fluorouracil','fluorouracil':'fluorouracil','taxol':'paclitaxel'}

if not WEB_GDSC_FILES:
    raise FileNotFoundError(f'No cellhit_gdsc*.csv exports found in {WEB_GDSC_DIR}')
print('New GDSC exports:')
for f in WEB_GDSC_FILES:
    print(f'  {f.name}: {f.stat().st_size/1e6:.1f} MB')
print('Existing matched cache:', OLD_CACHE_DB, 'exists:', OLD_CACHE_DB.is_file())
print('Original raw download fallback:', {k:p.exists() for k,p in ORIGINAL_INPUTS.items()})
print('New outputs:', OUTPUT_DIR)

## Read clinical MSI/TMB labels

Match predictions to clinical groups by TCGA patient barcode.

In [ ]:
GROUPS = ['MSI-H', 'MSS/TMB-H', 'MSS/TMB-L']
CANON = {'MSIH':'MSI-H', 'MSI-H':'MSI-H', 'MSS/TMB-H':'MSS/TMB-H',
         'MSS/TMB-L':'MSS/TMB-L'}

def patient_id(series):
    return series.astype('string').str.upper().str.strip().str.extract(
        r'^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})(?:$|[-_.])', expand=False
    )

def read_clinical(fp, cancer):
    if not fp.is_file():
        raise FileNotFoundError(f'Missing clinical file: {fp}')
    header = pd.read_csv(fp, nrows=0).columns
    wanted = {'Tumor_Sample_Barcode','sample_submitter_id', 'case_submitter_id',
              'MSI_TMB', 'dataset', 'project_id'}
    use = [c for c in header if c in wanted]
    df = pd.read_csv(fp, usecols=use, low_memory=False)
    if 'MSI_TMB' not in df:
        raise ValueError(f'No MSI_TMB column in {fp}')
    barcode_col = next((c for c in ['Tumor_Sample_Barcode','sample_submitter_id','case_submitter_id']
                        if c in df), None)
    if barcode_col is None:
        raise ValueError(f'No TCGA barcode column in {fp}')
    if cancer == 'CRC' and 'dataset' in df:
        df = df[df['dataset'].astype(str).str.upper().isin(['TCGA-COAD','TCGA-READ'])].copy()
    if cancer == 'UCEC' and 'project_id' in df:
        df = df[df['project_id'].astype(str).str.upper().eq('TCGA-UCEC')].copy()
    df['patient_id'] = patient_id(df[barcode_col])
    df['group_label'] = df['MSI_TMB'].astype('string').str.strip().str.upper().map(CANON)
    df = df.dropna(subset=['patient_id','group_label'])
    conflicts = df.groupby('patient_id')['group_label'].nunique()
    if (conflicts > 1).any():
        raise ValueError(f'Conflicting group labels within {fp.name}: {conflicts[conflicts>1].index.tolist()[:10]}')
    return df[['patient_id','group_label']].drop_duplicates('patient_id').assign(cancer=cancer)

clinical = pd.concat([read_clinical(CRC_CLINICAL,'CRC'),
                      read_clinical(UCEC_CLINICAL,'UCEC')], ignore_index=True)
conflicting_cancers = clinical.groupby('patient_id')['cancer'].nunique()
if (conflicting_cancers > 1).any():
    raise ValueError('CRC/UCEC clinical entries overlap in TCGA patient ID')
clinical.to_csv(OUTPUT_DIR / 'clinical_patient_labels.csv',index=False)
print(pd.crosstab(clinical['cancer'],clinical['group_label']).reindex(columns=GROUPS, fill_value=0))
print('Distinct clinically labeled TCGA patients:',clinical.patient_id.nunique())
assert set(clinical.group_label).issubset(GROUPS)

## Build the prediction cache

Deduplicate repeated sample–drug rows. Average distinct aliquots within each patient and screening dataset.

In [ ]:
GROUPS = ['MSI-H', 'MSS/TMB-H', 'MSS/TMB-L']
READ_COLS_OLD = ['DrugName','DrugID','Source','SampleIndex','Predictions','PredictionsStd',
                 'QuantileScore','ExperimentalMedian','ModelMSE','ModelCorr','PutativeTarget']
READ_COLS_WEB = ['DrugName','DrugID','index','prediction','std','QuantileScore','DrugMedian',
                 'PutativeTarget','dataset']
CACHE_COLS = ['dataset','patient_id','cancer','group_label','drug_id','drug_name','drug_key',
              'target','prediction','pred_std','quantile_score','exp_median','model_mse',
              'model_corr','n_aliquots','prediction_origin']

def normalize_drug(name):
    val = re.sub(r'\s+', ' ', str(name).strip().lower())
    return DRUG_ALIASES.get(val, val)

def file_signature(path):
    stat = path.stat()
    return {'path':str(path.resolve()),'size':stat.st_size,'mtime_ns':stat.st_mtime_ns}

def get_manifest():
    sources = WEB_GDSC_FILES + [CRC_CLINICAL,UCEC_CLINICAL]
    if OLD_CACHE_DB.is_file():
        sources += [OLD_CACHE_DB]
    else:
        sources += list(ORIGINAL_INPUTS.values())
    return {'files':[file_signature(p) for p in sources],
            'clinical_sha256':hashlib.sha256(clinical.sort_values('patient_id').to_csv(index=False).encode()).hexdigest(),
            'drug_aliases':DRUG_ALIASES}

def cache_valid():
    if REBUILD_CACHE or not CACHE_DB.is_file() or not CACHE_MANIFEST.is_file(): return False
    try:
        with sqlite3.connect(CACHE_DB) as con:
            tables = {x[0] for x in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        return CACHE_MANIFEST.read_text() == json.dumps(get_manifest(),sort_keys=True,indent=2) and 'patient_predictions' in tables
    except Exception:
        return False

def init_tables(con):
    con.executescript("""
    CREATE TABLE clinical_labels(patient_id TEXT PRIMARY KEY,cancer TEXT,group_label TEXT);
    CREATE TABLE patient_predictions (
       dataset TEXT,patient_id TEXT,cancer TEXT,group_label TEXT,drug_id TEXT,
       drug_name TEXT,drug_key TEXT,target TEXT,prediction REAL,pred_std REAL,
       quantile_score REAL,exp_median REAL,model_mse REAL,model_corr REAL,
       n_aliquots INTEGER,prediction_origin TEXT);
    """)
    clinical[['patient_id','cancer','group_label']].to_sql('clinical_labels',con,if_exists='append',index=False)

def load_old_from_existing_cache(con):
    with sqlite3.connect(OLD_CACHE_DB) as check:
        cols = {x[1] for x in check.execute('PRAGMA table_info(patient_predictions)')}
    required = {'dataset','patient_id','drug_id','drug_name','prediction','quantile_score','model_corr'}
    if not required.issubset(cols):
        raise RuntimeError(f'Existing old cache missing columns {required-cols}; delete old cache or use old CSV fallback')
    con.execute('ATTACH DATABASE ? AS old_src',(str(OLD_CACHE_DB),))
    try:
        con.execute("""
            INSERT INTO patient_predictions
            SELECT p.dataset,p.patient_id,l.cancer,l.group_label,p.drug_id,p.drug_name,
                   p.drug_key,p.target,p.prediction,p.pred_std,p.quantile_score,p.exp_median,
                   p.model_mse,p.model_corr,p.n_aliquots,'published_cache'
            FROM old_src.patient_predictions p
            JOIN clinical_labels l ON p.patient_id=l.patient_id
            WHERE p.dataset IN ('GDSC','PRISM')
        """)
        n=con.execute('SELECT COUNT(*) FROM patient_predictions').fetchone()[0]
    finally:
        con.commit()
        con.execute('DETACH DATABASE old_src')
    print(f'Loaded {n:,} old patient–drug records from previous SQLite cache (no full CSV scan)')

def load_old_from_csv(con):

    for db, path in ORIGINAL_INPUTS.items():
        if not path.is_file(): raise FileNotFoundError(f'Old cache missing; also cannot find {path}')
        n_all=n_kept=0
        for ch in pd.read_csv(path, usecols=lambda col:col in READ_COLS_OLD, chunksize=CHUNK_SIZE,
                              dtype={'DrugID':'string','DrugName':'string','SampleIndex':'string'},low_memory=False):
            n_all+=len(ch)
            req={'DrugID','DrugName','SampleIndex','Predictions','QuantileScore'}
            if not req.issubset(ch): raise ValueError(f'{path.name} lacks columns {req-set(ch)}')
            if 'Source' in ch: ch=ch[ch.Source.astype('string').str.upper().eq('TCGA')]
            ch=ch[ch.SampleIndex.astype('string').str.match(r'^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}(?:$|[-_.])',na=False)].copy()
            ch['patient_id']=patient_id(ch.SampleIndex)
            ch=ch.merge(clinical,how='inner',on='patient_id')
            if ch.empty:continue
            out=pd.DataFrame({'dataset':db,'patient_id':ch.patient_id,'cancer':ch.cancer,
                'group_label':ch.group_label,'drug_id':ch.DrugID.astype(str),
                'drug_name':ch.DrugName.astype(str),'drug_key':ch.DrugName.map(normalize_drug),
                'target':ch.get('PutativeTarget',pd.Series(index=ch.index,dtype=str)),
                'prediction':pd.to_numeric(ch.Predictions,errors='coerce'),
                'pred_std':pd.to_numeric(ch.get('PredictionsStd',pd.Series(index=ch.index,dtype=float)),errors='coerce'),
                'quantile_score':pd.to_numeric(ch.QuantileScore,errors='coerce'),
                'exp_median':pd.to_numeric(ch.get('ExperimentalMedian',pd.Series(index=ch.index,dtype=float)),errors='coerce'),
                'model_mse':pd.to_numeric(ch.get('ModelMSE',pd.Series(index=ch.index,dtype=float)),errors='coerce'),
                'model_corr':pd.to_numeric(ch.get('ModelCorr',pd.Series(index=ch.index,dtype=float)),errors='coerce'),
                'n_aliquots':1,'prediction_origin':'published_csv'})
            out=out.dropna(subset=['drug_id','prediction','quantile_score'])
            out[CACHE_COLS].to_sql('old_raw_staging',con,index=False,if_exists='append')
            n_kept+=len(out)
        print(db,'old CSV scanned',f'{n_all:,}','kept',f'{n_kept:,}')
    con.execute("""
    INSERT INTO patient_predictions
    SELECT dataset,patient_id,MIN(cancer),MIN(group_label),drug_id,
           MIN(drug_name),MIN(drug_key),MIN(target),AVG(prediction),AVG(pred_std),
           AVG(quantile_score),AVG(exp_median),AVG(model_mse),AVG(model_corr),
           COUNT(*),'published_csv'
    FROM old_raw_staging GROUP BY dataset,patient_id,drug_id
    """)
    con.execute('DROP TABLE old_raw_staging')

def load_new_gdsc_web(con):
    audit=[]
    n_all_drug_ids=set()
    for f in WEB_GDSC_FILES:
        rows=kept=0
        for ch in pd.read_csv(f, usecols=lambda x:x in READ_COLS_WEB, chunksize=CHUNK_SIZE,
                              dtype={'DrugID':'string','DrugName':'string','index':'string'},low_memory=False):
            rows+=len(ch)
            required={'DrugID','DrugName','index','prediction','QuantileScore'}
            if not required.issubset(ch): raise ValueError(f'{f.name} lacks columns {required-set(ch)}')
            if 'dataset' in ch:
                bad=ch['dataset'].dropna().astype(str).str.upper().ne('GDSC')
                if bad.any(): raise ValueError(f'{f.name} has dataset other than GDSC')
            ch=ch[ch['index'].str.match(r'^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}(?:$|[-_.])',na=False)].copy()
            if ch.empty: continue
            ch['patient_id']=patient_id(ch['index'])
            ch=ch.merge(clinical,how='inner',on='patient_id')
            if ch.empty: continue
            out=pd.DataFrame({'patient_id':ch.patient_id,'cancer':ch.cancer,'group_label':ch.group_label,
                'drug_id':ch.DrugID.astype(str),'drug_name':ch.DrugName.astype(str),
                'drug_key':ch.DrugName.map(normalize_drug),'target':ch.get('PutativeTarget',pd.Series(index=ch.index,dtype=str)),
                'sample_index':ch['index'].astype(str),
                'prediction':pd.to_numeric(ch.prediction,errors='coerce'),
                'pred_std':pd.to_numeric(ch.get('std',pd.Series(index=ch.index,dtype=float)),errors='coerce'),
                'quantile_score':pd.to_numeric(ch.QuantileScore,errors='coerce'),
                'exp_median':pd.to_numeric(ch.get('DrugMedian',pd.Series(index=ch.index,dtype=float)),errors='coerce'),
                'source_file':f.name})
            out=out.dropna(subset=['drug_id','prediction','quantile_score'])
            if ((out.quantile_score < 0) | (out.quantile_score>1)).any():
                raise ValueError(f'QuantileScore values outside 0–1 in {f.name}')
            n_all_drug_ids.update(out.drug_id.unique().tolist())
            out.to_sql('web_staging',con,if_exists='append',index=False)
            kept+=len(out)
        audit.append(dict(source_file=f.name,rows_scanned=rows,rows_matching_clinical=kept))
        print(f'{f.name}: {rows:,} rows scanned; {kept:,} predictions matched to clinical patients')
    pd.DataFrame(audit).to_csv(OUTPUT_DIR/'new_gdsc_export_file_audit.csv',index=False)
    print('Distinct drug IDs in web exports:',len(n_all_drug_ids))
    if len(n_all_drug_ids) < 10:
        warnings.warn('FEWER THAN 10 GDSC DRUGS in web exports: check that the CellHit export was NOT filtered to one drug.')

    con.execute('CREATE INDEX idx_web_sample ON web_staging(patient_id,drug_id,sample_index)')
    n_raw=con.execute('SELECT COUNT(*) FROM web_staging').fetchone()[0]
    con.execute("""
      CREATE TABLE web_dedup AS
      SELECT patient_id,cancer,group_label,drug_id,drug_name,drug_key,target,sample_index,
             prediction,pred_std,quantile_score,exp_median
      FROM (
        SELECT *,ROW_NUMBER() OVER (PARTITION BY patient_id,drug_id,sample_index ORDER BY rowid DESC) AS rn
        FROM web_staging
      ) WHERE rn=1
    """)
    n_unique=con.execute('SELECT COUNT(*) FROM web_dedup').fetchone()[0]
    print('Repeated export rows removed:',n_raw-n_unique)
    con.execute("""
      CREATE TABLE web_patient AS
      SELECT patient_id,MIN(cancer) AS cancer,MIN(group_label) AS group_label,drug_id,
             MIN(drug_name) AS drug_name,MIN(drug_key) AS drug_key,MIN(target) AS target,
             AVG(prediction) AS prediction,AVG(pred_std) AS pred_std,
             AVG(quantile_score) AS quantile_score,AVG(exp_median) AS exp_median,
             COUNT(*) AS n_aliquots
      FROM web_dedup GROUP BY patient_id,drug_id
    """)
    print('Unique new web patient–drug predictions:',con.execute('SELECT COUNT(*) FROM web_patient').fetchone()[0])
    con.execute('DROP TABLE web_staging')
    con.execute('DROP TABLE web_dedup')
    con.execute('CREATE INDEX idx_web_pd ON web_patient(patient_id,drug_id)')

def merge_new_over_old(con):

    id_conflicts=pd.read_sql_query("""
       SELECT o.drug_id,o.drug_key AS published_drug,w.drug_key AS web_drug
       FROM (SELECT drug_id,MIN(drug_key) AS drug_key FROM patient_predictions
             WHERE dataset='GDSC' GROUP BY drug_id) o
       JOIN (SELECT drug_id,MIN(drug_key) AS drug_key FROM web_patient
             GROUP BY drug_id) w ON o.drug_id=w.drug_id
       WHERE o.drug_key<>w.drug_key
    """,con)
    id_conflicts.to_csv(OUTPUT_DIR/'gdsc_drug_id_name_conflicts.csv',index=False)
    if len(id_conflicts):
        raise ValueError('The same GDSC DrugID names a different compound in old vs web data. '
                         'Refusing to merge by DrugID; inspect GDSC_DRUG_ID_NAME_CONFLICTS.csv. '
                         f'Example: {id_conflicts.head(3).to_dict("records")}')
    con.execute('CREATE INDEX idx_old_pd ON patient_predictions(dataset,patient_id,drug_id)')
    qc=pd.read_sql_query("""
    SELECT p.cancer,p.group_label,COUNT(*) AS n_overlapping_patient_drugs,
           AVG(ABS(p.prediction-w.prediction)) AS mean_abs_prediction_change,
           AVG(ABS(p.quantile_score-w.quantile_score)) AS mean_abs_quantile_change,
           SUM(CASE WHEN lower(trim(p.drug_name))<>lower(trim(w.drug_name)) THEN 1 ELSE 0 END) AS name_difference_count
    FROM patient_predictions p JOIN web_patient w
     ON p.dataset='GDSC' AND p.patient_id=w.patient_id AND p.drug_id=w.drug_id
    GROUP BY p.cancer,p.group_label
    """,con)
    qc.to_csv(OUTPUT_DIR/'overlapping_old_vs_web_gdsc.csv',index=False)
    if not qc.empty: display(qc)

    con.execute("""
      DELETE FROM patient_predictions
      WHERE dataset='GDSC' AND EXISTS
      (SELECT 1 FROM web_patient w WHERE w.patient_id=patient_predictions.patient_id
          AND w.drug_id=patient_predictions.drug_id)
    """)
    con.execute("""
      INSERT INTO patient_predictions
      SELECT 'GDSC',patient_id,cancer,group_label,drug_id,drug_name,drug_key,target,
             prediction,pred_std,quantile_score,exp_median,NULL,NULL,n_aliquots,'new_web_export'
      FROM web_patient
    """)
    con.execute('CREATE UNIQUE INDEX idx_combined_unique ON patient_predictions(dataset,patient_id,drug_id)')
    con.execute('CREATE INDEX idx_combined_group ON patient_predictions(dataset,cancer,group_label,drug_id)')
    con.execute('CREATE INDEX idx_combined_patient ON patient_predictions(dataset,patient_id)')
    con.execute('DROP TABLE web_patient')
    con.commit()

if cache_valid():
    print('Reusing updated SQLite cache:',CACHE_DB)
else:
    print('Creating the updated prediction cache...')
    t0=time.time()

    if CACHE_DB.exists(): CACHE_DB.unlink()
    with sqlite3.connect(CACHE_DB) as con:
        con.execute('PRAGMA journal_mode=DELETE')
        con.execute('PRAGMA synchronous=NORMAL')
        init_tables(con)
        if OLD_CACHE_DB.is_file(): load_old_from_existing_cache(con)
        else: load_old_from_csv(con)
        before=pd.read_sql_query("""SELECT dataset,cancer,group_label,COUNT(DISTINCT patient_id) AS n_patients
           FROM patient_predictions GROUP BY dataset,cancer,group_label""",con)
        before['version']='original'
        load_new_gdsc_web(con)
        merge_new_over_old(con)
        after=pd.read_sql_query("""SELECT dataset,cancer,group_label,COUNT(DISTINCT patient_id) AS n_patients
           FROM patient_predictions GROUP BY dataset,cancer,group_label""",con)
        after['version']='merged'
        pd.concat([before,after],ignore_index=True).to_csv(OUTPUT_DIR/'patient_coverage_before_after.csv',index=False)
        print('Coverage before/after (patient counts):')
        display(pd.concat([before,after],ignore_index=True).sort_values(['cancer','group_label','dataset','version']))
    CACHE_MANIFEST.write_text(json.dumps(get_manifest(),sort_keys=True,indent=2))
    print('Merged cache ready after',round(time.time()-t0,1),'seconds')

with sqlite3.connect(CACHE_DB) as con:
    availability = pd.read_sql_query("""
      SELECT dataset,cancer,group_label,COUNT(DISTINCT patient_id) AS n_patients,
      COUNT(DISTINCT drug_id) AS n_drugs,COUNT(*) AS n_patient_drug_predictions,
      SUM(CASE WHEN n_aliquots>1 THEN 1 ELSE 0 END) AS n_multi_aliquot
      FROM patient_predictions GROUP BY dataset,cancer,group_label""",con)
    by_origin = pd.read_sql_query("""SELECT dataset,prediction_origin,COUNT(*) AS n_patient_drug_predictions,
         COUNT(DISTINCT patient_id) AS n_patients FROM patient_predictions
         GROUP BY dataset,prediction_origin""",con)
availability.to_csv(OUTPUT_DIR/'matched_samples_and_drugs.csv',index=False)
by_origin.to_csv(OUTPUT_DIR/'prediction_source_breakdown.csv',index=False)
print('Updated group coverage:')
display(availability)
print('Where the final predictions came from:')
display(by_origin)